In [4]:
import os
os.chdir(r"C:\Users\xАлександра\Саша\rulebased-concept-tree")

In [3]:
import subprocess
 
exe = r"C:\Users\xАлександра\Саша\rulebased-concept-tree\tree_tagger\bin\tree-tagger.exe"
par = r"C:\Users\xАлександра\Саша\rulebased-concept-tree\tree_tagger\lib\russian.par"
 
text = "Я\nдал\nкнигу\nсестре\n.\n"

result = subprocess.run(
    [exe, par, "-token", "-lemma", "-sgml"],
    input=text.encode('utf-8'),
    capture_output=True
)
 
print(result.stdout.decode('utf-8'))

Я	P-1-snn	я
дал	Vmis-sma-p	дать
книгу	Ncfsan	книга
сестре	Ncfsdy	сестра
.	SENT	.



In [7]:
import os
 
# ❗ старый путь (как в проекте)
old_path = r"C:/Users/userboy/Desktop/Саша/rulebased-concept-tree"
 
# ❗ новый путь (твой Windows)
new_path = r"C:/Users/xАлександра/Саша/rulebased-concept-tree"
 
root_dir = new_path  # где лежит проект
 
changed_files = []
 
for subdir, _, files in os.walk(root_dir):
    for file in files:
        if file.endswith((".py", ".txt", ".json")):
            path = os.path.join(subdir, file)
 
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
 
            if old_path in content:
                content = content.replace(old_path, new_path.replace("\\", "/"))
 
                with open(path, "w", encoding="utf-8") as f:
                    f.write(content)
 
                changed_files.append(path)
 
print("Изменено файлов:", len(changed_files))
for f in changed_files:
    print(f)

Изменено файлов: 5
C:/Users/xАлександра/Саша/rulebased-concept-tree\prepare — копия (2).py
C:/Users/xАлександра/Саша/rulebased-concept-tree\prepare — копия.py
C:/Users/xАлександра/Саша/rulebased-concept-tree\prepare.py
C:/Users/xАлександра/Саша/rulebased-concept-tree\run_maltparser.py
C:/Users/xАлександра/Саша/rulebased-concept-tree\corpus\download_micusp.py


In [5]:
!python pipeline.py --lang ru --input input.conll

Running MaltParser...



Cleaning text: 100%|##########| 6429/6429 [00:00<00:00, 13928.95it/s]

Processing sentences: 100%|##########| 22866/22866 [11:38:21<00:00,  1.83s/it]
-----------------------------------------------------------------------------
                          MaltParser 1.9.2                             
-----------------------------------------------------------------------------
         MALT (Models and Algorithms for Language Technology) Group          
             Vaxjo University and Uppsala University                         
                             Sweden                                          
-----------------------------------------------------------------------------

Started: Thu Apr 30 00:50:10 MSK 2026
  Transition system    : Arc-Eager
  Parser configuration : Nivre with allow_root=true, allow_reduce=false and enforce_tree=false
  Data Format          : /russian/conllx.xml
.          	      1	      2s	    447MB
.          	     10	      4s	    587MB
.          	    1

In [ ]:
from run_maltparser import annotate

annotate("input.conll", "output_full.conll", lang="ru")

In [8]:
import os

if os.path.exists("output_full.conll"):
    size = os.path.getsize("output_full.conll") / 1e9
    print(f"✅ ФАЙЛ СУЩЕСТВУЕТ! Размер: {size:.2f} ГБ")
    print(f"Путь: {os.path.abspath('output_full.conll')}")
else:
    print("❌ Файла нет, придется запустить парсинг заново")
    print("(но это маловероятно, судя по логу)")

✅ ФАЙЛ СУЩЕСТВУЕТ! Размер: 6.92 ГБ
Путь: C:\Users\xАлександра\Саша\rulebased-concept-tree\output_full.conll


In [9]:
from make_graph_ru import build_ru_graph
from ru_rb_anaphora import resolve_anaphora_ru
from graph.graph import visualize_graph_interactive

resolved = resolve_anaphora_ru("output_full.conll")
g = build_ru_graph(resolved)
visualize_graph_interactive(g, output="graph.html")

print("Граф построен!")

KeyboardInterrupt: 

In [4]:
from resolve_anaphora_limited import resolve_anaphora_ru_limited

# Запускаем на 3000 предложениях
resolved = resolve_anaphora_ru_limited("output_full.conll", max_sentences=500)

print(f"✅ Обработано {len(resolved)} предложений")

Загрузка первых 500 предложений из output_full.conll...
Чтение файла: 1639068 строк [00:03, 445225.34 строк/s]
Загружено 500 предложений. Обработка анафоры...
Обработка анафоры: 100%|████████████████████████████████████████████████████| 500/500 [10:38:55<00:00, 76.67s/ предл.]
✅ Обработано 500 предложений.
✅ Обработано 500 предложений


In [5]:
from make_graph_ru import build_ru_graph
from graph.graph import visualize_graph_interactive

# Строим граф на уже обработанных предложениях
g = build_ru_graph(resolved)  # resolved из предыдущей ячейки

# Визуализируем
visualize_graph_interactive(g, output="graph_500.html")

print("✅ Готово!")

Interactive graph saved to graph_500.html
✅ Готово!


In [16]:
from metrics import calculate_metrics
import pandas as pd

metrics = calculate_metrics(g)
pd.DataFrame([metrics]).to_csv("graph_metrics.csv", index=False, encoding="utf-8")
print(f"Метрики сохранены в graph_metrics.csv")

ImportError: cannot import name 'calculate_metrics' from 'metrics' (unknown location)

In [20]:
import pandas as pd

def calculate_metrics_minimal(graph):
    """
    Метрики без NetworkX, только из объектов графа
    """
    n = len(graph.vertices)
    m = len(graph.edges)
    
    metrics = {
        "num_vertices": n,
        "num_edges": m,
        "average_degree": 2 * m / n if n > 0 else 0,
        "density": 2 * m / (n * (n - 1)) if n > 1 else 0,
    }
    
    # Считаем степени вручную
    if n > 0:
        in_degrees = {}
        out_degrees = {}
        
        for edge in graph.edges:
            out_degrees[edge.agent_1] = out_degrees.get(edge.agent_1, 0) + 1
            in_degrees[edge.agent_2] = in_degrees.get(edge.agent_2, 0) + 1
        
        total_in = sum(in_degrees.values())
        total_out = sum(out_degrees.values())
        
        metrics["average_in_degree"] = total_in / n
        metrics["average_out_degree"] = total_out / n
    
    return metrics

# Вычисляем
metrics = calculate_metrics_minimal(g)

# Сохраняем
pd.DataFrame([metrics]).to_csv("graph_metrics.csv", index=False, encoding="utf-8")

print("\nМЕТРИКИ ГРАФА СОХРАНЕНЫ В graph_metrics.csv")
print("="*50)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")


МЕТРИКИ ГРАФА СОХРАНЕНЫ В graph_metrics.csv
num_vertices: 68819
num_edges: 76455350
average_degree: 2221.9256
density: 0.0323
average_in_degree: 1110.9628
average_out_degree: 1110.9628


In [21]:
# 3. Вероятности ребер (из уже загруженного resolved)
from collections import Counter

# Считаем биграммы из resolved
bigram_counts = Counter()
unigram_counts = Counter()

for sent in resolved:
    words = [tok.form.lower() for tok in sent.tokens]
    for i in range(len(words)-1):
        w1, w2 = words[i], words[i+1]
        bigram_counts[(w1, w2)] += 1
        unigram_counts[w1] += 1
    if words:
        unigram_counts[words[-1]] += 1

# Считаем вероятности для ребер
edge_data = []
for edge in g.edges:
    w1, w2 = edge.agent_1.lower(), edge.agent_2.lower()
    prob = bigram_counts.get((w1, w2), 0) / unigram_counts.get(w1, 1)
    prob_rev = bigram_counts.get((w2, w1), 0) / unigram_counts.get(w2, 1)
    
    edge_data.append({
        "agent_1": edge.agent_1,
        "agent_2": edge.agent_2, 
        "meaning": edge.meaning,
        "probability_direct": prob,
        "probability_reverse": prob_rev,
        "is_unexpected": max(prob, prob_rev) < 0.01
    })

pd.DataFrame(edge_data).to_csv("edge_probabilities.csv", index=False, encoding="utf-8")
print(f"Сохранено {len(edge_data)} ребер в edge_probabilities.csv")

Сохранено 76455350 ребер в edge_probabilities.csv


In [23]:
import csv
from collections import Counter
import math

# Считаем t-score для всех биграмм из resolved
unigrams = Counter()
bigrams = Counter()
total_words = 0

print("Подсчет биграмм...")
for sent in resolved:  # используем уже загруженное
    words = [tok.form.lower() for tok in sent.tokens]
    for w in words:
        unigrams[w] += 1
        total_words += 1
    for i in range(len(words)-1):
        bigrams[(words[i], words[i+1])] += 1
print(f"Найдено {len(bigrams)} уникальных биграмм")

# Вычисляем t-score для биграмм
t_scores = {}
for (w1, w2), obs in bigrams.items():
    p1 = unigrams[w1] / total_words
    p2 = unigrams[w2] / total_words
    exp = p1 * p2 * total_words
    t_scores[(w1, w2)] = (obs - exp) / math.sqrt(obs) if obs > 0 else -exp

# Сохраняем результаты по частям (не в память)
print("Сохранение результатов...")

# Сохраняем все биграммы с t-score в CSV частями
with open("all_bigrams_t_scores.csv", "w", encoding="utf-8", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["word_1", "word_2", "t_score", "is_unexpected"])
    
    for (w1, w2), t_score in t_scores.items():
        writer.writerow([w1, w2, t_score, t_score < 0])

print(f"✅ Сохранено {len(t_scores)} биграмм в all_bigrams_t_scores.csv")

# Отдельно сохраняем только ребра графа (которых меньше)
print("Сохранение ребер графа...")
with open("edge_t_scores.csv", "w", encoding="utf-8", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["agent_1", "agent_2", "meaning", "t_score", "is_unexpected"])
    
    for edge in g.edges:
        w1, w2 = edge.agent_1.lower(), edge.agent_2.lower()
        
        # Ищем прямую и обратную биграмму
        t_direct = t_scores.get((w1, w2))
        t_reverse = t_scores.get((w2, w1))
        
        if t_direct is not None or t_reverse is not None:
            t_score = max([x for x in [t_direct, t_reverse] if x is not None])
        else:
            t_score = None
        
        writer.writerow([edge.agent_1, edge.agent_2, edge.meaning, t_score, t_score < 0 if t_score is not None else None])

print(f"✅ Сохранено ребер в edge_t_scores.csv")

Подсчет биграмм...
Найдено 695303 уникальных биграмм
Сохранение результатов...
✅ Сохранено 695303 биграмм в all_bigrams_t_scores.csv
Сохранение ребер графа...
✅ Сохранено ребер в edge_t_scores.csv


In [67]:
import csv
from collections import Counter
import math

# 1. Считаем биграммы из resolved (леммы)
unigrams = Counter()
bigrams = Counter()
total_words = 0

print("Подсчет биграмм из лемм...")
for sent in resolved:
    words = [tok.lemma.lower() for tok in sent.tokens if tok.lemma and tok.lemma != '_']
    for w in words:
        unigrams[w] += 1
        total_words += 1
    for i in range(len(words)-1):
        bigrams[(words[i], words[i+1])] += 1

print(f"Всего слов: {total_words}, уникальных биграмм: {len(bigrams)}")

# 2. Вычисляем t-score для биграмм
print("Вычисление t-score...")
t_scores = {}
for (w1, w2), obs in bigrams.items():
    if unigrams[w1] > 0:
        p1 = unigrams[w1] / total_words
        p2 = unigrams[w2] / total_words
        exp = p1 * p2 * total_words
        t_scores[(w1, w2)] = (obs - exp) / math.sqrt(obs) if obs > 0 else -exp

Подсчет биграмм из лемм...
Всего слов: 1638068, уникальных биграмм: 685231
Вычисление t-score...


In [68]:
import csv
from collections import Counter
import re

# 1. Считаем частоты слов и биграммы из resolved
print("Подсчет частот из обработанных текстов...")

unigrams = Counter()  # отдельные слова
bigrams = Counter()   # пары слов (подряд)
total_words = 0

for sent in resolved:
    # Получаем все слова в предложении
    words = []
    for tok in sent.tokens:
        word = tok.form.lower()
        word = re.sub(r'[^\w\-а-яё]', '', word)
        if word and len(word) > 1:
            words.append(word)
            unigrams[word] += 1
            total_words += 1
    
    # Считаем биграммы (последовательные пары)
    for i in range(len(words)-1):
        bigrams[(words[i], words[i+1])] += 1

print(f"Всего слов: {total_words}")
print(f"Уникальных слов: {len(unigrams)}")
print(f"Уникальных биграмм: {len(bigrams)}")

Подсчет частот из обработанных текстов...
Всего слов: 1205300
Уникальных слов: 58253
Уникальных биграмм: 773330


In [69]:
# 2. Функция для извлечения ВСЕХ значимых слов из концепта
def extract_words(concept):
    """
    Извлекает все слова из концепта
    """
    words = re.findall(r'\b[а-яё]+\b', concept.lower())
    # Очищаем и фильтруем короткие
    return [w for w in words if len(w) > 1]

# 3. Функция для вычисления вероятности между двумя концептами
def get_concept_probability(concept1, concept2):
    """
    Считает максимальную вероятность P(слово2|слово1) или P(слово1|слово2)
    между любыми словами из двух концептов
    """
    words1 = extract_words(concept1)
    words2 = extract_words(concept2)
    
    if not words1 or not words2:
        return 0, None, 'no_words'
    
    best_prob = 0
    best_pair = None
    best_type = None
    
    # Ищем прямые вероятности (w1 → w2)
    for w1 in words1:
        for w2 in words2:
            if w1 == w2:
                continue
            
            # Прямая биграмма
            count = bigrams.get((w1, w2), 0)
            if count > 0:
                prob = count / unigrams.get(w1, 1)
                if prob > best_prob:
                    best_prob = prob
                    best_pair = (w1, w2)
                    best_type = f"'{w1}' → '{w2}' (в тексте ровно подряд)"
    
    # Если не нашли ничего, пробуем обратную (w2 → w1)
    if best_prob == 0:
        for w2 in words2:
            for w1 in words1:
                if w1 == w2:
                    continue
                count = bigrams.get((w2, w1), 0)
                if count > 0:
                    prob = count / unigrams.get(w2, 1)
                    if prob > best_prob:
                        best_prob = prob
                        best_pair = (w2, w1)
                        best_type = f"'{w2}' → '{w1}' (в тексте ровно подряд)"
    
    # Если всё равно 0, пробуем совместную встречаемость в предложении
    if best_prob == 0:
        # Считаем, сколько раз оба слова встречаются в одном предложении
        cooccurrence = 0
        for sent in resolved:
            sent_words = set()
            for tok in sent.tokens:
                w = re.sub(r'[^\w\-а-яё]', '', tok.form.lower())
                if w and len(w) > 1:
                    sent_words.add(w)
            
            has_any1 = any(w in sent_words for w in words1)
            has_any2 = any(w in sent_words for w in words2)
            
            if has_any1 and has_any2:
                cooccurrence += 1
        
        if cooccurrence > 0:
            best_prob = cooccurrence / len(resolved)
            best_pair = (words1[0], words2[0])  # просто для информации
            best_type = f"встречаются в одном предложении ({cooccurrence} раз)"
    
    return best_prob, best_pair, best_type

In [70]:
# 4. Проверяем на примерах
print("\n" + "="*60)
print("ПРОВЕРКА НА ПРИМЕРАХ")
print("="*60)

test_pairs = [
    ("кошка", "мышка"),
    ("безысходный", "бедность"),
    ("градус", "человек"),
    ("безысходный бедность", "градус"),
    ("одиночество", "градус"),
    ("кот", "ест"),
]

for c1, c2 in test_pairs:
    prob, pair, ptype = get_concept_probability(c1, c2)
    print(f"\n{c1} → {c2}")
    print(f"  вероятность: {prob:.6f}" if prob > 0 else "  вероятность: 0")
    if prob > 0:
        print(f"  найдено: {ptype}")


ПРОВЕРКА НА ПРИМЕРАХ

кошка → мышка
  вероятность: 0.016000
  найдено: встречаются в одном предложении (8 раз)

безысходный → бедность
  вероятность: 0.200000
  найдено: 'безысходный' → 'бедность' (в тексте ровно подряд)

градус → человек
  вероятность: 0.030000
  найдено: встречаются в одном предложении (15 раз)

безысходный бедность → градус
  вероятность: 0.008000
  найдено: встречаются в одном предложении (4 раз)

одиночество → градус
  вероятность: 0.012000
  найдено: встречаются в одном предложении (6 раз)

кот → ест
  вероятность: 0


In [37]:
df = pd.read_csv("graph_metrics.csv")
df.head()

,num_vertices,num_edges,average_degree,density,average_in_degree,average_out_degree
0,68819,76455350,2221.925631,0.032287,1110.962816,1110.962816


In [39]:
df = pd.read_csv("edge_t_scores.csv", nrows=100)
df.head()

,agent_1,agent_2,meaning,t_score,is_unexpected
0,безысходный бедность,градус,быть по,NaN,NaN
1,остроконечный полоса,градус,быть по,NaN,NaN
2,одиночество,градус,быть по,NaN,NaN
3,ребенок,градус,быть по,NaN,NaN
4,сам,градус,быть по,NaN,NaN


In [59]:
df = pd.read_csv("all_bigrams_t_scores.csv")
df.sample(10)

,word_1,word_2,t_score,is_unexpected
493577,грек,торжествовать,0.999139,False
196587,правильный,отделение,0.997112,False
467258,говорить,надеяться,0.576399,False
556664,губонинский,белый,0.999528,False
155607,бесталанный,серебряный,0.999504,False
317061,встать,минуте,0.956962,False
644206,идти,негодование,1.287780,False
156119,то,поясница,0.990428,False
515106,запивать,мясо,0.999814,False
240223,обидеть,зверь,0.991814,False


In [76]:
df = pd.read_csv("edge_final_probabilities.csv", nrows=100)
df.head(20)

,agent_1,agent_2,meaning,probability,match_type,best_pair,is_unexpected
0,безысходный бедность,градус,быть по,0.008,в одном предложении (4 раз),бедность→градус,True
1,остроконечный полоса,градус,быть по,0.020,в одном предложении (10 раз),полоса→градус,False
2,одиночество,градус,быть по,0.012,в одном предложении (6 раз),одиночество→градус,False
3,ребенок,градус,быть по,0.026,в одном предложении (13 раз),ребенок→градус,False
4,сам,градус,быть по,0.002,в одном предложении (1 раз),сам→градус,True
5,купчиха,градус,быть по,0.006,в одном предложении (3 раз),купчиха→градус,True
6,душа,градус,быть по,0.028,в одном предложении (14 раз),душа→градус,False
7,сын,градус,быть по,0.024,в одном предложении (12 раз),сын→градус,False
8,самый,градус,быть по,0.002,в одном предложении (1 раз),самый→градус,True
9,он,градус,быть по,0.002,в одном предложении (1 раз),он→градус,True


In [2]:
import re
import pandas as pd
from collections import Counter

# Ваш анекдот
joke = "Зять, а ты знать, где найти тот мужчина, который спасти я, когда я тонуть?- Да, он уже приходить к я извиняться!"

# 1.1 Извлекаем биграммы из анекдота
def extract_bigrams_from_text(text):
    """Извлекает биграммы из текста"""
    # Очищаем и токенизируем
    words = re.findall(r'\b[а-яё]+\b', text.lower())
    
    # Создаем биграммы
    bigrams = []
    for i in range(len(words)-1):
        bigrams.append(f"{words[i]} → {words[i+1]}")
    
    return bigrams, words

joke_bigrams, joke_words = extract_bigrams_from_text(joke)
for i, bg in enumerate(joke_bigrams, 1):
    print(f"{i}. {bg}")

1. зять → а
2. а → ты
3. ты → знать
4. знать → где
5. где → найти
6. найти → тот
7. тот → мужчина
8. мужчина → который
9. который → спасти
10. спасти → я
11. я → когда
12. когда → я
13. я → тонуть
14. тонуть → да
15. да → он
16. он → уже
17. уже → приходить
18. приходить → к
19. к → я
20. я → извиняться


In [5]:
# 1.2 Ищем эти биграммы в edge_final_probabilities.csv
try:
    df_edges = pd.read_csv("edge_final_probabilities.csv")
    
    print("\n" + "="*60)
    print("НАЛИЧИЕ БИГРАММ АНЕКДОТА В ГРАФЕ")
    print("="*60)
    
    found_bigrams = []
    for bg in joke_bigrams:
        # Разбиваем биграмму на слова
        w1, w2 = bg.split(' → ')
        
        # Ищем в ребрах графа
        mask = ((df_edges['agent_1'].str.lower() == w1) & 
                (df_edges['agent_2'].str.lower() == w2))
        
        matching = df_edges[mask]
        
        if len(matching) > 0:
            prob = matching.iloc[0]['probability']
            found_bigrams.append((bg, prob))
            print(f"✅ {bg}: вероятность = {prob:.4f}")
        else:
            # Проверяем обратную биграмму (w2 → w1)
            mask_rev = ((df_edges['agent_1'].str.lower() == w2) & 
                        (df_edges['agent_2'].str.lower() == w1))
            matching_rev = df_edges[mask_rev]
            
            if len(matching_rev) > 0:
                prob = matching_rev.iloc[0]['probability']
                found_bigrams.append((bg, prob))
                print(f"🔄 {bg} (обратная): вероятность = {prob:.4f}")
            else:
                print(f"❌ {bg}: не найдено в графе")
    
    print(f"\nНайдено в графе: {len(found_bigrams)} из {len(joke_bigrams)} биграмм")
    
except FileNotFoundError:
    print("Файл edge_final_probabilities.csv не найден")


НАЛИЧИЕ БИГРАММ АНЕКДОТА В ГРАФЕ
❌ зять → а: не найдено в графе
❌ а → ты: не найдено в графе
❌ ты → знать: не найдено в графе
❌ знать → где: не найдено в графе
❌ где → найти: не найдено в графе
❌ найти → тот: не найдено в графе
🔄 тот → мужчина (обратная): вероятность = 0.0020
❌ мужчина → который: не найдено в графе
❌ который → спасти: не найдено в графе
❌ спасти → я: не найдено в графе
❌ я → когда: не найдено в графе
❌ когда → я: не найдено в графе
❌ я → тонуть: не найдено в графе
❌ тонуть → да: не найдено в графе
❌ да → он: не найдено в графе
❌ он → уже: не найдено в графе
❌ уже → приходить: не найдено в графе
❌ приходить → к: не найдено в графе
❌ к → я: не найдено в графе
❌ я → извиняться: не найдено в графе

Найдено в графе: 1 из 20 биграмм


In [6]:
# 2.1 Загружаем t-score для всех биграмм
try:
    df_t_scores = pd.read_csv("all_bigrams_t_scores.csv")
    
    print("\n" + "="*60)
    print("T-SCORE БИГРАММ АНЕКДОТА")
    print("="*60)
    
    for bg in joke_bigrams:
        w1, w2 = bg.split(' → ')
        
        # Ищем в t-scores
        mask = (df_t_scores['word_1'] == w1) & (df_t_scores['word_2'] == w2)
        matching = df_t_scores[mask]
        
        if len(matching) > 0:
            t_score = matching.iloc[0]['t_score']
            is_unexpected = matching.iloc[0]['is_unexpected']
            print(f"✅ {bg}: t-score = {t_score:.4f}, неожиданная: {is_unexpected}")
        else:
            # Проверяем обратную
            mask_rev = (df_t_scores['word_1'] == w2) & (df_t_scores['word_2'] == w1)
            matching_rev = df_t_scores[mask_rev]
            
            if len(matching_rev) > 0:
                t_score = matching_rev.iloc[0]['t_score']
                is_unexpected = matching_rev.iloc[0]['is_unexpected']
                print(f"🔄 {bg} (обратная): t-score = {t_score:.4f}, неожиданная: {is_unexpected}")
            else:
                print(f"❌ {bg}: не найдено в t-scores")
                
except FileNotFoundError:
    print("Файл all_bigrams_t_scores.csv не найден")


T-SCORE БИГРАММ АНЕКДОТА
❌ зять → а: не найдено в t-scores
❌ а → ты: не найдено в t-scores
❌ ты → знать: не найдено в t-scores
❌ знать → где: не найдено в t-scores
❌ где → найти: не найдено в t-scores
❌ найти → тот: не найдено в t-scores
❌ тот → мужчина: не найдено в t-scores
❌ мужчина → который: не найдено в t-scores
❌ который → спасти: не найдено в t-scores
❌ спасти → я: не найдено в t-scores
❌ я → когда: не найдено в t-scores
❌ когда → я: не найдено в t-scores
❌ я → тонуть: не найдено в t-scores
❌ тонуть → да: не найдено в t-scores
❌ да → он: не найдено в t-scores
❌ он → уже: не найдено в t-scores
🔄 уже → приходить (обратная): t-score = 0.8938, неожиданная: False
✅ приходить → к: t-score = 7.1643, неожиданная: False
❌ к → я: не найдено в t-scores
❌ я → извиняться: не найдено в t-scores


In [20]:
import prepare
from run_maltparser import annotate
import os

# Ваш анекдот
joke_text = "Зять, а ты знаешь, где найти того мужчину, который спас меня, когда я тонула?- Да, он уже приходил ко мне извиняться!"

# 1. Создаем CoNLL файл без синтаксиса (только POS)
joke_conll_raw = "joke_raw.conll"
prepare.write_conllx_file(joke_text, filename=joke_conll_raw, lang='ru')
print(f"✅ Создан файл с POS-тегами: {joke_conll_raw}")

# 2. Запускаем MaltParser для получения синтаксических связей
print("\nЗапуск MaltParser для анекдота...")
try:
    # ВАЖНО: используем annotate с правильным output
    annotate(joke_conll_raw, "joke_parsed.conll", lang='ru')
    print("✅ MaltParser завершил обработку")
except Exception as e:
    print(f"Ошибка: {e}")
    print("\nПроверьте, что файл russian.mco существует в папке maltparser")

✅ Создан файл с POS-тегами: joke_raw.conll

Запуск MaltParser для анекдота...
✅ MaltParser завершил обработку


In [21]:
# 3. Проверяем результат парсинга
print("\n" + "="*60)
print("ПРОВЕРКА РЕЗУЛЬТАТА ПАРСИНГА")
print("="*60)

with open("joke_parsed.conll", "r", encoding="utf-8") as f:
    lines = f.readlines()[:30]  # первые 30 строк

print("Первые строки обработанного файла:")
for line in lines[:20]:
    print(line.strip())

# Проверяем, есть ли HEAD не равный 0
print("\n" + "="*60)
has_relations = False
with open("joke_parsed.conll", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip() and not line.startswith('#'):
            parts = line.split('\t')
            if len(parts) >= 8:
                head = parts[6]
                if head != '0' and head != '_':
                    has_relations = True
                    break

if has_relations:
    print("✅ Синтаксические связи найдены!")
else:
    print("❌ Синтаксических связей нет (все head=0)")


ПРОВЕРКА РЕЗУЛЬТАТА ПАРСИНГА
Первые строки обработанного файла:
1	Зять	Зять	Ncmsny	Ncmsny	_	0	ROOT	_	_
2	,	,	,	,	_	1	PUNC	_	_
3	а	а	C	C	_	1	сент-соч	_	_
4	ты	ты	P-2-snn	P-2-snn	_	5	огранич	_	_
5	знаешь	знаешь	Vmip2s-a-e	Vmip2s-a-e	_	3	соч-союзн	_	_
6	,	,	,	,	_	5	PUNC	_	_
7	где	где	P-----r	P-----r	_	10	предик	_	_
8	найти	найти	Vmn----a-p	Vmn----a-p	_	9	огранич	_	_
9	того	того	P--msaa	P--msaa	_	10	огранич	_	_
10	мужчину	мужчину	Ncmsay	Ncmsay	_	5	релят	_	_
11	,	,	,	,	_	10	PUNC	_	_
12	который	который	P--msna	P--msna	_	14	предик	_	_
13	спас	спас	Vmis-sma-p	Vmis-sma-p	_	14	огранич	_	_
14	меня	меня	P-1-san	P-1-san	_	10	релят	_	_
15	,	,	,	,	_	14	PUNC	_	_
16	когда	когда	C	C	_	14	обст	_	_
17	я	я	P-1-snn	P-1-snn	_	18	огранич	_	_
18	тонула	тонула	Vmis-sfa-e	Vmis-sfa-e	_	16	подч-союзн	_	_
19	?	?	SENT	SENT	_	18	PUNC	_	_
20	-	-	-	-	_	19	PUNC	_	_

✅ Синтаксические связи найдены!


In [22]:
# 4. Теперь строим граф из обработанного файла
from sent_class import parse_conll
from make_graph_ru import build_ru_graph

# Читаем синтаксически размеченный файл
sentences = parse_conll("joke_parsed.conll")
print(f"Загружено предложений: {len(sentences)}")

# Выводим токены с синтаксическими связями
if sentences:
    print("\nТокены с синтаксическими связями:")
    print(f"{'ID':<5} {'FORM':<15} {'HEAD':<6} {'DEPREL':<15}")
    print("-"*50)
    for tok in sentences[0].tokens:
        print(f"{tok.id:<5} {tok.form:<15} {tok.head:<6} {tok.deprel:<15}")

# Строим граф
g_joke = build_ru_graph(sentences)

print(f"\nВершины графа: {len(g_joke.vertices)}")
print(f"Ребра графа: {len(g_joke.edges)}")

print("\nРебра:")
for e in g_joke.edges:
    print(f"  {e.agent_1} --[{e.meaning}]--> {e.agent_2}")

Загружено предложений: 1

Токены с синтаксическими связями:
ID    FORM            HEAD   DEPREL         
--------------------------------------------------
1     Зять            0      ROOT           
2     ,               1      PUNC           
3     а               1      сент-соч       
4     ты              5      огранич        
5     знаешь          3      соч-союзн      
6     ,               5      PUNC           
7     где             10     предик         
8     найти           9      огранич        
9     того            10     огранич        
10    мужчину         5      релят          
11    ,               10     PUNC           
12    который         14     предик         
13    спас            14     огранич        
14    меня            10     релят          
15    ,               14     PUNC           
16    когда           14     обст           
17    я               18     огранич        
18    тонула          16     подч-союзн     
19    ?               18     PUNC 

In [ ]:
import re
import pandas as pd
import json

# Читаем HTML файл
with open("graph_500_corpus.html", "r", encoding="utf-8") as f:
    html_content = f.read()

# 1. Извлекаем вершины
nodes_match = re.search(r'var nodes = new vis\.DataSet\((.*?)\);', html_content, re.DOTALL)
if nodes_match:
    nodes_str = nodes_match.group(1)
    # Преобразуем строку в Python список
    # Заменяем null на None, true на True, false на False
    nodes_str = nodes_str.replace('null', 'None').replace('true', 'True').replace('false', 'False')
    nodes = eval(nodes_str)
    print(f"✅ Извлечено вершин: {len(nodes)}")
else:
    print("❌ Вершины не найдены")
    nodes = []

# 2. Извлекаем ребра
edges_match = re.search(r'var edges = new vis\.DataSet\((.*?)\);', html_content, re.DOTALL)
if edges_match:
    edges_str = edges_match.group(1)
    edges_str = edges_str.replace('null', 'None').replace('true', 'True').replace('false', 'False')
    edges = eval(edges_str)
    print(f"✅ Извлечено ребер: {len(edges)}")
else:
    print("❌ Ребра не найдены")
    edges = []